# Model-Based Sepsis RL

---

## Setup

In [1]:
import autograd.numpy as np
import autograd.numpy.random as npr
import pandas as pd
from autograd.misc.optimizers import adam
from bayesian_neural_net import make_nn_funs
from PPO import PPO

Device set to : cpu


Load training data and combine validation and test sets. Use state features from `state_features.txt`.

In [ ]:
state_features = open('state_features.txt').read().splitlines()
train_df = pd.read_csv('rl_train_data_final_cont.csv')
val_df = pd.read_csv('rl_val_data_final_cont.csv')
test_df = pd.read_csv('rl_test_data_final_cont.csv')
eval_df = pd.concat([val_df, test_df], ignore_index=True)
state_dim = len(state_features)
action_dim = 2
print('Train samples:', len(train_df))
print('Eval samples:', len(eval_df))
print('Number of state features:', state_dim)

Train samples: 171481
Eval samples: 73057
Number of state features: 48


Helper function to build transitions where the next state is the following bloc in the same ICU stay, matching the logic used in `cql_q_network.ipynb`.

In [5]:

import numpy as onp

def build_transitions(df):
    states, actions, rewards, next_states, done_flags = [], [], [], [], []
    for i in range(len(df)):
        cur_state = df.loc[i, state_features]
        action = df.loc[i, ['vaso_input', 'iv_input']]
        reward = df.loc[i, 'reward']
        if i != len(df) - 1 and df.loc[i, 'icustayid'] == df.loc[i + 1, 'icustayid']:
            next_state = df.loc[i + 1, state_features]
            done = 0
        else:
            next_state = onp.zeros(len(state_features))
            done = 1
        states.append(cur_state.values)
        actions.append(action.values)
        rewards.append(reward)
        next_states.append(next_state.values)
        done_flags.append(done)
    return (onp.array(states), onp.array(actions), onp.array(rewards), onp.array(next_states), onp.array(done_flags))


## Bayesian Neural Network Environment

In [7]:
train_states, train_actions, train_rewards, train_next_states, train_done = build_transitions(train_df)
mask = train_done == 0
inputs = onp.hstack([train_states[mask], train_actions[mask]])
state_deltas = train_next_states[mask] - train_states[mask]
print('Transitions used for training:', inputs.shape[0])
print('Input dimension:', inputs.shape[1])

layer_sizes = [inputs.shape[1], 32, 32, state_deltas.shape[1]]
L2_reg, noise_var = 1.0, 0.1
num_weights, pred_fun, logprob = make_nn_funs(layer_sizes, L2_reg, noise_var)
from autograd import grad

def black_box_vi(logprob, num_weights, num_samples=20):
    rs = npr.RandomState(0)
    def objective(params, t):
        mean, log_std = onp.split(params, 2)
        samples = mean + onp.exp(log_std) * rs.randn(num_samples, num_weights)
        log_q = -0.5*onp.sum(((samples-mean)/onp.exp(log_std))**2 + 2*log_std + onp.log(2*onp.pi), axis=1)
        log_p = logprob(samples, t)
        return onp.mean(log_q - log_p)
    gradient = grad(objective)
    def unpack(params):
        mean, log_std = onp.split(params, 2)
        return mean, log_std
    return objective, gradient, unpack

log_posterior = lambda w, t: logprob(w, inputs, state_deltas)
objective, gradient, unpack_params = black_box_vi(log_posterior, num_weights)
rs = npr.RandomState(0)
init_mean = rs.randn(num_weights)
init_log_std = -5*onp.ones(num_weights)
init_params = onp.concatenate([init_mean, init_log_std])
variational_params = adam(gradient, init_params, step_size=0.01, num_iters=100)
mean, log_std = unpack_params(variational_params)
print('Trained BNN with', num_weights, 'parameters')


AttributeError: 'numpy.ndarray' object has no attribute 'values'

Evaluate model on the combined validation and test sets.

In [ ]:

eval_states, eval_actions, _, eval_next_states, eval_done = build_transitions(eval_df)
eval_mask = eval_done == 0
eval_inputs = onp.hstack([eval_states[eval_mask], eval_actions[eval_mask]])
true_deltas = eval_next_states[eval_mask] - eval_states[eval_mask]
weights = mean[None, :]
pred_deltas = pred_fun(weights, eval_inputs)[0]
mse = onp.mean((pred_deltas - true_deltas)**2)
print('Evaluation transitions:', eval_inputs.shape[0])
print('MSE on val+test:', mse)


## Policy Search with PPO

In [ ]:

class SepsisBNNEnv:
    def __init__(self, mean, log_std, pred_fun):
        self.mean = mean
        self.log_std = log_std
        self.pred_fun = pred_fun
        self.rs = npr.RandomState(0)
        self.reset()
    def reset(self):
        self.idx = self.rs.randint(0, len(train_states))
        self.state = train_states[self.idx]
        return self.state
    def step(self, action):
        weights = self.mean + onp.exp(self.log_std) * self.rs.randn(1, len(self.mean))
        delta = self.pred_fun(weights, onp.hstack([self.state, action])[None,:])[0]
        next_state = self.state + delta
        reward = train_rewards[self.idx]
        done = bool(train_done[self.idx])
        self.idx = (self.idx + 1) % len(train_states)
        self.state = next_state
        return next_state, reward, done, {}

env = SepsisBNNEnv(mean, log_std, pred_fun)
ppo = PPO(state_dim=state_dim, action_dim=action_dim, lr_actor=3e-4, lr_critic=1e-3, gamma=0.99, K_epochs=5, eps_clip=0.2, has_continuous_action_space=True)
for episode in range(2):
    state = env.reset()
    for t in range(5):
        action = ppo.select_action(state)
        next_state, reward, done, _ = env.step(action)
        ppo.buffer.rewards.append(reward)
        ppo.buffer.is_terminals.append(done)
        state = next_state
        if done:
            print('Episode', episode, 'terminated at step', t)
            break
    ppo.update()
print('PPO training loop finished')


## Combining PPO with Clinician Policy based on SOFA

Placeholder for blending strategies that rely on clinician policy when SOFA indicates low-confidence regions.